# 04 – The data pile: bucket, SQL catalog and basic concepts

**Goal (Task 3, part 1):** create a **bucket for the pile** and a **SQL catalog** (SQLite) with PyIceberg.

| Task 2: data **lake** | Task 3: data **pile** (lakehouse) |
|---|---|
| loose Parquet files in a bucket | **Iceberg tables** in a bucket |
| "a table" = a folder of files | a table = data files **+ metadata files** that say which files belong to it |
| nobody keeps track of the files | a **catalog** keeps the list of tables |
| no versions, no schema history | **snapshots** (time travel) and **schema evolution** |

**Basic concepts**

- **Catalog** → the "phone book" of the pile. For every table it stores **where its newest metadata file is**. Ours is a **SQL catalog** = a small **SQLite** database file.
- **Namespace** → a group of tables, like a schema in PostgreSQL. We use it for the **medallion zones**: `bronze` (raw copies of the sources, this task), later `silver` (cleaned) and `gold` (ready for reports).
- **Table** → `bronze.postgres_animals`. Its files live in the bucket: `data/` (Parquet with the rows) and `metadata/` (JSON + Avro files that describe the table).
- **Snapshot** → a version of a table. Every write creates a new snapshot; old ones are kept.
- **Schema** → the list of columns; Iceberg remembers every version of it.

In [ ]:
from pile import (PILES, BRONZE, catalog_properties, get_catalog,
                  ensure_pile_bucket, list_pile, show_catalog_db)
from lake import _fs

## Step 1 – The pile bucket

Create the bucket **`pawhome-pile`** in the **Silo console** (http://localhost:9001 → Buckets → Create Bucket), next to the Task 2 bucket `pawhome-lake`.
The cell below lists the buckets and only creates `pawhome-pile` if you haven't done it in the console yet.

In [ ]:
ensure_pile_bucket("silo")
print("Buckets in Silo:", _fs("silo").ls(""))

## Step 2 – The SQL catalog (SQLite)

`get_catalog("silo")` calls PyIceberg's `load_catalog(...)` with these settings:

- `type: sql` → a SQL catalog
- `uri` → the **SQLite file** that stores the catalog (in `work/pile_catalog/`, so it is kept)
- `warehouse` → the **bucket** where the table files are written
- `s3.*` → how to reach Silo

In [ ]:
props = catalog_properties("silo")
for key, value in props.items():
    print(f"{key:22} {'*****' if 'secret' in key else value}")

catalog = get_catalog("silo")
print("\nCatalog:", catalog.name, type(catalog).__name__)

## Step 3 – The `bronze` namespace (medallion zone)

In [ ]:
catalog.create_namespace_if_not_exists(BRONZE)
print("Namespaces:", catalog.list_namespaces())
print("Tables in bronze:", catalog.list_tables(BRONZE))

## Step 4 – What is inside the SQLite catalog?

PyIceberg created two tables in the SQLite file. `iceberg_tables` gets **one row per Iceberg table** with the location of its current metadata file. It is still empty – the tables are created by dlt in notebooks 05 and 06. Come back and rerun this cell afterwards.

In [ ]:
import sqlite3
path = props["uri"].replace("sqlite:///", "")
with sqlite3.connect(path) as con:
    print("SQLite tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])
show_catalog_db("silo")

In [ ]:
list_pile("")      # the files in the pile bucket (empty until notebooks 05 / 06 run)

## Bonus – pile and catalog in the additional object store (RustFS)

Same steps for RustFS: its own bucket `pawhome-pile` and its own SQLite catalog `pawhome_rustfs.db`.

In [ ]:
ensure_pile_bucket("rustfs")
catalog_rustfs = get_catalog("rustfs")
catalog_rustfs.create_namespace_if_not_exists(BRONZE)
print("RustFS catalog:", catalog_rustfs.name, "→", catalog_properties("rustfs")["warehouse"])
print("Namespaces:", catalog_rustfs.list_namespaces())